[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itorque2024/capstone-ninjavan/blob/main/notebooks/03_fraud_detection.ipynb)

# 03 — Fraud Detection
**NinjaVan Operations Intelligence Suite**

Isolation Forest (unsupervised) + LightGBM (supervised) — flags suspicious shipping claims.

---
- **Run in:** Google Colab (CPU is fine)
- **Input:** `ninjavan_optionB_datasets/fraud_data.csv`
- **Output:** `src/models/fraud_model.pkl` → used by `src/agents/fraud_agent.py`
- **AI types covered:** ML (Isolation Forest unsupervised anomaly detection + LightGBM supervised classifier)


In [ ]:
# Install packages not available by default on Colab
!pip install -q lightgbm scikit-learn pandas numpy matplotlib seaborn joblib


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from lightgbm import LGBMClassifier


In [ ]:
import os

if not os.path.exists('capstone-ninjavan'):
    !git clone https://github.com/itorque2024/capstone-ninjavan.git

if os.path.basename(os.getcwd()) != 'capstone-ninjavan':
    os.chdir('capstone-ninjavan')

print('Working dir:', os.getcwd())

## 1. Load & Explore Data

In [ ]:
raw = pd.read_csv('ninjavan_optionB_datasets/fraud_data.csv')

print(f'Shape: {raw.shape}')
print(f'Columns: {list(raw.columns)}')
print(f'Fraud rate: {raw["fraud_flag"].mean()*100:.2f}%')
print(f'Missing: {raw.isnull().sum().to_dict()}')
raw.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Parcel value distribution
axes[0].hist(raw['parcel_value'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Parcel Value Distribution')
axes[0].set_xlabel('Value (SGD)')

# Value by fraud flag
axes[1].boxplot(
    [raw.loc[raw['fraud_flag']==0, 'parcel_value'],
     raw.loc[raw['fraud_flag']==1, 'parcel_value']],
    labels=['Legitimate', 'Fraud']
)
axes[1].set_title('Parcel Value: Fraud vs Legitimate')
axes[1].set_ylabel('Value (SGD)')

# Class balance
counts = raw['fraud_flag'].value_counts()
axes[2].pie(counts, labels=['Legitimate', 'Fraud'], colors=['steelblue', 'red'],
            autopct='%1.1f%%', startangle=90)
axes[2].set_title('Class Balance')

plt.tight_layout()
plt.show()

## 2. Feature Engineering

Derive value-based signals from . Behavioural signals (, , ) are already present in the dataset — no simulation needed.

In [ ]:
df = raw.copy()

# Value-based derived features
df["value_log"]      = np.log1p(df["parcel_value"])
df["value_zscore"]   = (df["parcel_value"] - df["parcel_value"].mean()) / df["parcel_value"].std()
df["value_pct"]      = df["parcel_value"].rank(pct=True)
df["is_high_value"]  = (df["parcel_value"] > df["parcel_value"].quantile(0.95)).astype(int)
df["is_very_low"]    = (df["parcel_value"] < df["parcel_value"].quantile(0.02)).astype(int)

# Behavioural features now come directly from the CSV (no simulation)
# prior_claims, account_age_days, claim_lag_days are real columns

FEATURES = [
    "parcel_value",
    "value_log",
    "value_zscore",
    "value_pct",
    "is_high_value",
    "is_very_low",
    "prior_claims",
    "account_age_days",
    "claim_lag_days",
]

print("Feature preview:")
df[FEATURES + ["fraud_flag"]].head()


In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df[FEATURES + ['fraud_flag']].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0)
plt.title('Fraud Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Model A — Isolation Forest (Unsupervised)

Isolation Forest detects anomalies without needing labeled fraud data. It scores each claim by how easily it can be "isolated" from the rest — outliers are isolated with fewer splits.

In [ ]:
X = df[FEATURES]
y = df['fraud_flag']

# Isolation Forest trains on ALL data (unsupervised — no labels)
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=float(y.mean()),  # expected fraud rate
    random_state=42,
    n_jobs=-1,
)
iso_forest.fit(X)

# decision_function: negative = more anomalous
iso_scores = iso_forest.decision_function(X)
iso_pred   = (iso_forest.predict(X) == -1).astype(int)  # -1 = anomaly = fraud

print('=== Isolation Forest (unsupervised) ===')
print(classification_report(y, iso_pred, target_names=['Legitimate', 'Fraud']))

In [ ]:
# Anomaly score distribution
plt.figure(figsize=(10, 4))
plt.hist(iso_scores[y==0], bins=50, alpha=0.6, label='Legitimate', color='steelblue')
plt.hist(iso_scores[y==1], bins=50, alpha=0.6, label='Fraud', color='red')
plt.axvline(0, color='black', linestyle='--', label='Decision boundary')
plt.title('Isolation Forest Anomaly Score Distribution')
plt.xlabel('Anomaly Score (more negative = more anomalous)')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Model B — LightGBM (Supervised)

LightGBM leverages labeled fraud cases for higher precision on known fraud patterns. Faster than XGBoost on large datasets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()  # handle imbalance

lgbm = LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=scale_pos,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
lgbm.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[],
)

lgbm_pred = lgbm.predict(X_test)
lgbm_prob = lgbm.predict_proba(X_test)[:, 1]

print('=== LightGBM (supervised) ===')
print(classification_report(y_test, lgbm_pred, target_names=['Legitimate', 'Fraud']))
print(f'ROC-AUC: {roc_auc_score(y_test, lgbm_prob):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Precision-Recall curve
prec, rec, thresholds = precision_recall_curve(y_test, lgbm_prob)
ap = average_precision_score(y_test, lgbm_prob)
axes[0].plot(rec, prec, color='steelblue', label=f'LightGBM (AP={ap:.3f})')
axes[0].set_title('Precision-Recall Curve')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].legend()

# Feature importance
feat_imp = pd.Series(lgbm.feature_importances_, index=FEATURES).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('LightGBM Feature Importance')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 5. Threshold Tuning

Default 0.5 threshold is not optimal for fraud. We tune to maximise F1 on the fraud class — balancing false negatives (missed fraud = costly) vs false positives (wrongful rejection = bad CX).

In [ ]:
from sklearn.metrics import f1_score

best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.2, 0.8, 0.01):
    preds = (lgbm_prob >= t).astype(int)
    f1 = f1_score(y_test, preds, pos_label=1, zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f'Optimal threshold: {best_t:.2f} → Fraud F1: {best_f1:.3f}')

tuned_pred = (lgbm_prob >= best_t).astype(int)
print('\n=== LightGBM at Optimal Threshold ===')
print(classification_report(y_test, tuned_pred, target_names=['Legitimate', 'Fraud']))

## 6. Save Model

Uses the `FraudDetectionModel` wrapper from `src/models/wrappers.py` — importable by `fraud_agent.py`:
```python
scores = model.decision_function(df)   # Isolation Forest anomaly scores
flags  = model.predict_fraud(df)       # LightGBM binary predictions
```
Input DataFrame must contain `parcel_value`, `prior_claims`, `account_age_days`, `claim_lag_days`.

In [ ]:
import sys
sys.path.insert(0, ".")
from src.models.wrappers import FraudDetectionModel

TRAIN_STATS = {
    "value_mean": float(df["parcel_value"].mean()),
    "value_std":  float(df["parcel_value"].std()),
    "p95":        float(df["parcel_value"].quantile(0.95)),
    "p02":        float(df["parcel_value"].quantile(0.02)),
}

fraud_model = FraudDetectionModel(
    iso_forest   = iso_forest,
    lgbm_model   = lgbm,
    feature_cols = FEATURES,
    train_stats  = TRAIN_STATS,
    threshold    = best_t,
)

# Sanity check
test_claims = pd.DataFrame([
    {"parcel_value": 25.0,  "prior_claims": 0, "account_age_days": 500, "claim_lag_days": 10},
    {"parcel_value": 450.0, "prior_claims": 6, "account_age_days": 5,   "claim_lag_days": 1},
])
scores = fraud_model.decision_function(test_claims)
flags  = fraud_model.predict_fraud(test_claims)
print("Sanity check:")
print(f"  Normal (25 SGD)  → score: {scores[0]:.3f} | flag: {flags[0]}")
print(f"  Fraud  (450 SGD) → score: {scores[1]:.3f} | flag: {flags[1]}")


In [ ]:
os.makedirs('src/models', exist_ok=True)
joblib.dump(fraud_model, 'src/models/fraud_model.pkl')
print('Saved → src/models/fraud_model.pkl')

loaded = joblib.load('src/models/fraud_model.pkl')
print(f'Load OK — scores: {loaded.decision_function(test_claims).round(3).tolist()}')

In [ ]:
# Model saved — commit to GitHub so HF Spaces has it:
#   git add src/models/
#   git commit -m 'Add trained model'
#   git push
print("Done. Commit and push src/models/ to GitHub.")

## Summary

| Model | Approach | Strength |
|-------|----------|----------|
| Isolation Forest | Unsupervised anomaly detection | Catches novel/unknown fraud patterns |
| LightGBM | Supervised classifier | High precision on known fraud patterns |
| **Ensemble** | IF scores + LightGBM | **Used in production — covers both** |

**Next:** Run `04_rag_chatbot.ipynb` to build the customer knowledge base.